In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *
import re
from datetime import datetime

p_schema = StructType([
    StructField('product_id',IntegerType()),
    StructField('product_name',StringType()),
    StructField('category',StringType()),
    StructField('brand',StringType()),
    StructField('price',IntegerType()),
    StructField('status',StringType()),
])

files = dbutils.fs.ls('/Volumes/madstagescd2/madstage/madstage/Initialfile/')

productsfiles = []

for file in files:
    match = re.search(r'(\d{4}-\d{2}-\d{2})',file.path)
    if file.name.startswith('product'):
        if match:
            file_time = datetime.strptime(match.group(1),"%Y-%m-%d")
            productsfiles.append((file_time,file.path))

if productsfiles:
    latest_file = max(productsfiles,key=lambda x:x[0])
    print(latest_file[0],latest_file[1])
    products = (spark.read.format('csv').option('header','true').schema(p_schema).load(latest_file[1]))

products = products.dropna(subset=('product_id'))
products = products.dropDuplicates()
products.fillna({
    'product_name':'Unknown',
    'category':'Unknown',
    'brand':'Unknown',
    'price':0,
    'status':'Unknown',
})

products = products.withColumn('product_name',f.initcap(f.trim(f.col('product_name')))).withColumn('category',f.initcap(f.trim(f.col('category')))).withColumn('brand',f.initcap(f.trim(f.col('brand')))).withColumn('price',f.round(f.col('price'),2)).withColumn('status',f.initcap(f.trim(f.col('status'))))

products = products.withColumn('dataValidation',f.when((f.col('status')=='Active')&(f.col('price')>=0),'Pass').otherwise('Fail'))

products = products.withColumn('is_current',f.lit('Y')).withColumn('updated_dt',f.lit('').cast('string')).withColumn('created_dt',f.current_date().cast('string'))

products = products.filter(f.col('dataValidation')=='Pass').drop('dataValidation')

products.show()

products.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('madstagescd2.madstage.silver_products')




